# VitalSense — ECG Model Training (PTB-XL)

**Prerequisites:**
1. Download PTB-XL from https://physionet.org/content/ptb-xl/1.0.3/
2. Unzip into `backend/data/ptbxl/` so that `ptbxl_database.csv` is at `backend/data/ptbxl/ptbxl_database.csv`
3. Use the `records100/` folder (100Hz)

**Est. time:** ~20 min on GPU (RTX 5060 / similar)  
**Output:** `backend/models/ecg_resnet.pt`

In [1]:
# IMPORTANT: torch must be imported FIRST on Windows (DLL load order fix)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import os, warnings
warnings.filterwarnings("ignore")

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
MODELS_DIR   = NOTEBOOK_DIR
DATA_DIR     = os.path.join(NOTEBOOK_DIR, "..", "data", "ptbxl")

if not os.path.exists(os.path.join(DATA_DIR, "ptbxl_database.csv")):
    raise FileNotFoundError(f"PTB-XL not found at {DATA_DIR}")

print(f"PTB-XL found at: {os.path.abspath(DATA_DIR)}")


PTB-XL found at: D:\Projects\DevHub\vitalsense\backend\data\ptbxl


In [2]:
import numpy as np
import pandas as pd
import wfdb
import ast

from sklearn.metrics import roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
SUPERCLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']

# ── Load metadata ─────────────────────────────────────────────────────────
Y = pd.read_csv(os.path.join(DATA_DIR, 'ptbxl_database.csv'), index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(ast.literal_eval)

agg_df = pd.read_csv(os.path.join(DATA_DIR, 'scp_statements.csv'), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1.0]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic:
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)
print(f'Records loaded: {len(Y)}')

Device: cuda


Records loaded: 21799


In [3]:
# ── Load signals (100Hz) ──────────────────────────────────────────────────
print('Loading signals (this takes a few minutes)...')
def load_raw_data(df, path):
    data = [wfdb.rdsamp(os.path.join(path, f))[0] for f in df.filename_lr]
    return np.array(data)  # (N, 1000, 12)

X_ecg = load_raw_data(Y, DATA_DIR)  # (N, 1000, 12)
print(f'Signals shape: {X_ecg.shape}')

# Encode labels (multi-label)
def encode_labels(Y_df):
    labels = np.zeros((len(Y_df), 5), dtype=np.float32)
    for i, row in enumerate(Y_df['diagnostic_superclass']):
        for cls in row:
            if cls in SUPERCLASSES:
                labels[i, SUPERCLASSES.index(cls)] = 1.0
    return labels

y_ecg = encode_labels(Y)
print(f'Label shape: {y_ecg.shape}')

# PTB-XL built-in folds: 1-8 train, 9 val, 10 test
train_mask = Y.strat_fold <= 8
val_mask   = Y.strat_fold == 9
test_mask  = Y.strat_fold == 10

# Reshape signals to (N, 12, 1000) for Conv1d
X_tr  = torch.FloatTensor(X_ecg[train_mask].transpose(0,2,1))
X_val = torch.FloatTensor(X_ecg[val_mask].transpose(0,2,1))
X_te  = torch.FloatTensor(X_ecg[test_mask].transpose(0,2,1))
y_tr  = torch.FloatTensor(y_ecg[train_mask])
y_val = torch.FloatTensor(y_ecg[val_mask])
y_te  = torch.FloatTensor(y_ecg[test_mask])
print(f'Train: {len(X_tr)}, Val: {len(X_val)}, Test: {len(X_te)}')

Loading signals (this takes a few minutes)...


Signals shape: (21799, 1000, 12)
Label shape: (21799, 5)


Train: 17418, Val: 2183, Test: 2198


In [4]:
# ── Model ─────────────────────────────────────────────────────────────────
class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=5, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, stride=stride, padding=kernel_size//2)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=kernel_size//2)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.relu  = nn.ReLU()
        self.drop  = nn.Dropout(0.2)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride=stride),
                nn.BatchNorm1d(out_ch)
            )
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)

class ECGResNet(nn.Module):
    def __init__(self, num_classes=5, num_leads=12):
        super().__init__()
        self.input_conv = nn.Conv1d(num_leads, 64, kernel_size=15, padding=7)
        self.bn0   = nn.BatchNorm1d(64)
        self.relu  = nn.ReLU()
        self.layer1 = ResBlock1D(64,  64)
        self.layer2 = ResBlock1D(64,  128, stride=2)
        self.layer3 = ResBlock1D(128, 256, stride=2)
        self.layer4 = ResBlock1D(256, 256, stride=2)
        self.pool   = nn.AdaptiveAvgPool1d(1)
        self.fc     = nn.Linear(256, num_classes)
        self.sig    = nn.Sigmoid()
    def forward(self, x):
        x = self.relu(self.bn0(self.input_conv(x)))
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.pool(x).squeeze(-1)
        return self.sig(self.fc(x))

model = ECGResNet().to(DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Model parameters: 1,436,357


In [5]:
# ── Training ──────────────────────────────────────────────────────────────
BATCH_SIZE = 32
EPOCHS     = 50
LR         = 1e-3

train_dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)
criterion = nn.BCELoss()

best_auc  = -1.0
best_path = os.path.join(MODELS_DIR, 'ecg_resnet.pt')

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_dl:
            all_preds.append(model(xb.to(DEVICE)).cpu().numpy())
            all_labels.append(yb.numpy())
    preds  = np.concatenate(all_preds)
    labels_np = np.concatenate(all_labels)
    try:
        val_auc = roc_auc_score(labels_np, preds, average='macro')
    except Exception:
        val_auc = 0.0
    scheduler.step(val_auc * -1)  # ReduceLROnPlateau minimises

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), best_path)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={total_loss/len(train_dl):.4f}  val_AUC={val_auc:.4f}  best={best_auc:.4f}')

if not os.path.exists(best_path):
    torch.save(model.state_dict(), best_path)


Epoch   1/50  loss=0.3464  val_AUC=0.8939  best=0.8939


Epoch  10/50  loss=0.2378  val_AUC=0.9286  best=0.9286


Epoch  20/50  loss=0.2003  val_AUC=0.9236  best=0.9309


Epoch  30/50  loss=0.1466  val_AUC=0.9256  best=0.9309


Epoch  40/50  loss=0.1447  val_AUC=0.9250  best=0.9309


Epoch  50/50  loss=0.1437  val_AUC=0.9253  best=0.9309


In [6]:
# ── Test evaluation ───────────────────────────────────────────────────────
model.load_state_dict(torch.load(best_path, map_location='cpu', weights_only=True))
model.eval()
test_dl = DataLoader(TensorDataset(X_te, y_te), batch_size=BATCH_SIZE)
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        all_preds.append(model(xb.to(DEVICE)).cpu().numpy())
        all_labels.append(yb.numpy())
preds = np.concatenate(all_preds)
labels_np = np.concatenate(all_labels)
test_auc = roc_auc_score(labels_np, preds, average='macro')

print()
print('=== ECG Model Training Complete ===')
print(f'1D-ResNet on PTB-XL ({len(Y):,} ECGs)')
print(f'Test Macro-AUC: {test_auc:.3f}')
print(f'Model saved to: {best_path}')


=== ECG Model Training Complete ===
1D-ResNet on PTB-XL (21,799 ECGs)
Test Macro-AUC: 0.925
Model saved to: D:\Projects\DevHub\vitalsense\backend\models\ecg_resnet.pt


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, multilabel_confusion_matrix

threshold = 0.5
pred_labels = (preds >= threshold).astype(int)
subset_accuracy = accuracy_score(labels_np, pred_labels)
label_accuracy = (labels_np == pred_labels).mean()
f1_micro = f1_score(labels_np, pred_labels, average='micro', zero_division=0)
f1_macro = f1_score(labels_np, pred_labels, average='macro', zero_division=0)
mcm = multilabel_confusion_matrix(labels_np, pred_labels)

per_class = []
sensitivities = []
specificities = []
for idx, cls in enumerate(SUPERCLASSES):
    tn, fp, fn, tp = mcm[idx].ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else float('nan')
    specificity = tn / (tn + fp) if (tn + fp) else float('nan')
    acc = (tp + tn) / (tp + tn + fp + fn)
    f1 = f1_score(labels_np[:, idx], pred_labels[:, idx], zero_division=0)
    sensitivities.append(sensitivity)
    specificities.append(specificity)
    per_class.append((cls, acc, f1, sensitivity, specificity))

print('Overall metrics:')
print()
print(f'Subset accuracy: {subset_accuracy:.4f}')
print(f'Label accuracy: {label_accuracy:.4f}')
print(f'F1 micro: {f1_micro:.4f}')
print(f'F1 macro: {f1_macro:.4f}')
print(f'Sensitivity macro: {np.nanmean(sensitivities):.4f}')
print(f'Specificity macro: {np.nanmean(specificities):.4f}')
print()
print('Per class:')
print()
for cls, acc, f1, sensitivity, specificity in per_class:
    print(f'{cls}: accuracy {acc:.4f}, F1 {f1:.4f}, sensitivity {sensitivity:.4f}, specificity {specificity:.4f}')
print()
print('Context:')
print()
print(f'test samples: {len(labels_np)}')
print(f'saved model: {os.path.basename(best_path)}')
print(f'notebook test macro-AUC from training run: {test_auc:.3f}')
